# Aula Interativa no Google Colab: Lei de Ohm com gráficos interativos

Nesta aula, vamos trabalhar com um arquivo `.csv` contendo dados experimentais simulados da **Lei de Ohm**.

A proposta é ensinar, de forma didática:

- como carregar um arquivo `.csv` no Google Colab;
- como visualizar e explorar os dados;
- como construir **gráficos interativos**;
- como interpretar a relação entre **tensão**, **corrente** e **resistência**;
- como estimar a resistência elétrica de um resistor a partir dos dados.

---

## Contexto físico

A Lei de Ohm é dada por:

$$
V = R \cdot I
$$

onde:

- $V$ é a tensão elétrica (em volts);
- $I$ é a corrente elétrica (em ampères);
- $R$ é a resistência elétrica (em ohms).

Se o resistor for ôhmico, a relação entre tensão e corrente será **linear**.


## Caminho do arquivo

Nesta aula, vamos usar o seguinte caminho no Google Drive:

```python
/content/drive/MyDrive/Colab Notebooks/dados_fisica_lei_ohm_interativo_200_pontos.csv
```

Antes de executar, confirme que o arquivo CSV foi enviado para esta pasta.


In [1]:
# ============================================================
# 1) Importação das bibliotecas
# ============================================================
# pandas: leitura e manipulação de dados tabulares
# numpy: operações numéricas
# plotly.express e plotly.graph_objects: gráficos interativos
# sklearn: ajuste linear simples para estimar a resistência
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================================
# 2) Leitura do arquivo CSV
# ============================================================
# Aqui definimos o caminho completo do arquivo no Google Drive.
# Se o arquivo estiver exatamente neste local, a leitura ocorrerá
# normalmente.
# ============================================================

caminho_arquivo = "/content/drive/MyDrive/Colab Notebooks/dados_fisica_lei_ohm_interativo_200_pontos.csv"

df = pd.read_csv(caminho_arquivo)

# Exibir as 10 primeiras linhas
df.head(10)


,ponto,tensao_V,corrente_A,resistencia_ohm
0,1,0.200,0.00429,46.652
1,2,0.299,0.00636,47.098
2,3,0.399,0.00857,46.548
3,4,0.498,0.01085,45.950
4,5,0.598,0.01268,47.166
5,6,0.697,0.01479,47.166
6,7,0.797,0.01736,45.912
7,8,0.896,0.01929,46.465
8,9,0.996,0.02104,47.333
9,10,1.095,0.02350,46.621


## Entendendo as colunas

O arquivo contém:

- **ponto**: índice numérico de cada medição;
- **tensao_V**: tensão elétrica medida em volts;
- **corrente_A**: corrente elétrica medida em ampères;
- **resistencia_ohm**: resistência calculada ponto a ponto usando $R = V/I$.

Mesmo com pequenas variações experimentais, esperamos que a resistência permaneça aproximadamente constante.


In [5]:
# ============================================================
# 3) Informações gerais do conjunto de dados
# ============================================================
# Vamos verificar o tamanho da tabela, os tipos de dados e um
# resumo estatístico inicial.
# ============================================================

print("Dimensão da tabela:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

print("\nResumo estatístico:")
display(df.describe())


Dimensão da tabela: (200, 4)

Tipos de dados:
ponto                int64
tensao_V           float64
corrente_A         float64
resistencia_ohm    float64
dtype: object

Resumo estatístico:


,ponto,tensao_V,corrente_A,resistencia_ohm
count,200.000000,200.000000,200.000000,200.000000
mean,100.500000,10.100000,0.214883,47.037860
std,57.879185,5.758977,0.122680,0.656461
min,1.000000,0.200000,0.004290,45.157000
25%,50.750000,5.150000,0.109585,46.649250
50%,100.500000,10.100000,0.212405,47.003000
75%,150.250000,15.050000,0.321510,47.502250
max,200.000000,20.000000,0.423780,48.922000


In [6]:
# ============================================================
# 4) Primeiro gráfico interativo: tensão versus corrente
# ============================================================
# Este é o gráfico mais importante da aula.
# Se a Lei de Ohm estiver sendo obedecida, a relação V x I
# deve se comportar aproximadamente como uma reta.
# ============================================================

fig = px.scatter(
    df,
    x="corrente_A",
    y="tensao_V",
    hover_data=["ponto", "resistencia_ohm"],
    title="Lei de Ohm: Tensão em função da Corrente",
    labels={
        "corrente_A": "Current (A)",
        "tensao_V": "Voltage (V)"
    }
)

fig.update_traces(marker=dict(size=7))
fig.update_layout(template="plotly_white")
fig.show()


In [7]:
# ============================================================
# 5) Adicionando uma linha de ajuste linear ao gráfico
# ============================================================
# Como V = R * I, podemos ajustar uma reta do tipo:
#
#    V = a*I + b
#
# O coeficiente angular 'a' será uma estimativa da resistência.
# ============================================================

X = df[["corrente_A"]]
y = df["tensao_V"]

modelo = LinearRegression()
modelo.fit(X, y)

y_previsto = modelo.predict(X)

R_estimado = modelo.coef_[0]
intercepto = modelo.intercept_
r2 = r2_score(y, y_previsto)

print(f"Resistência estimada pelo ajuste linear: {R_estimado:.4f} ohms")
print(f"Intercepto da reta: {intercepto:.6f} V")
print(f"Coeficiente de determinação R²: {r2:.6f}")


Resistência estimada pelo ajuste linear: 46.9248 ohms
Intercepto da reta: 0.016660 V
Coeficiente de determinação R²: 0.999225


In [8]:
# ============================================================
# 6) Gráfico interativo com os pontos experimentais e a reta
# ============================================================
# Agora vamos construir um gráfico interativo mais completo,
# mostrando simultaneamente:
# - os dados experimentais;
# - a reta ajustada pelo modelo linear.
# ============================================================

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["corrente_A"],
    y=df["tensao_V"],
    mode="markers",
    name="Experimental data",
    text=df["ponto"],
    hovertemplate="Point: %{text}<br>Current: %{x:.4f} A<br>Voltage: %{y:.3f} V<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=df["corrente_A"],
    y=y_previsto,
    mode="lines",
    name="Linear fit",
    hovertemplate="Current: %{x:.4f} A<br>Fitted voltage: %{y:.3f} V<extra></extra>"
))

fig.update_layout(
    title="Lei de Ohm com ajuste linear interativo",
    xaxis_title="Current (A)",
    yaxis_title="Voltage (V)",
    template="plotly_white"
)

fig.show()


In [9]:
# ============================================================
# 7) Histograma interativo da resistência
# ============================================================
# Aqui observamos como os valores de resistência se distribuem.
# Em uma situação ideal, todos os valores seriam iguais.
# Na prática, pequenas diferenças podem ocorrer por ruído,
# arredondamento ou incertezas experimentais.
# ============================================================

fig = px.histogram(
    df,
    x="resistencia_ohm",
    nbins=25,
    title="Distribution of Resistance Values",
    labels={"resistencia_ohm": "Resistance (ohm)"}
)

fig.update_layout(template="plotly_white")
fig.show()


In [10]:
# ============================================================
# 8) Gráfico interativo da resistência ao longo das medições
# ============================================================
# Este gráfico ajuda a discutir estabilidade experimental.
# Se a resistência variar pouco, o resistor apresenta um
# comportamento coerente com o esperado.
# ============================================================

fig = px.line(
    df,
    x="ponto",
    y="resistencia_ohm",
    markers=True,
    title="Resistance calculated at each measurement point",
    labels={
        "ponto": "Measurement point",
        "resistencia_ohm": "Resistance (ohm)"
    }
)

fig.update_layout(template="plotly_white")
fig.show()


## Interpretação física dos resultados

A partir dos gráficos interativos, os alunos podem observar que:

1. **Tensão e corrente crescem juntas** de forma aproximadamente linear;
2. o **ajuste linear** representa bem os dados;
3. o **coeficiente angular da reta** corresponde à resistência elétrica;
4. a resistência calculada ponto a ponto oscila pouco em torno de um valor médio.

Isso caracteriza o comportamento esperado de um **resistor ôhmico**.


In [11]:
# ============================================================
# 9) Estatísticas finais da resistência
# ============================================================
# Vamos calcular medidas simples para resumir a resistência.
# ============================================================

media_R = df["resistencia_ohm"].mean()
desvio_R = df["resistencia_ohm"].std()
min_R = df["resistencia_ohm"].min()
max_R = df["resistencia_ohm"].max()

print(f"Mean resistance: {media_R:.4f} ohms")
print(f"Standard deviation: {desvio_R:.4f} ohms")
print(f"Minimum value: {min_R:.4f} ohms")
print(f"Maximum value: {max_R:.4f} ohms")


Mean resistance: 47.0379 ohms
Standard deviation: 0.6565 ohms
Minimum value: 45.1570 ohms
Maximum value: 48.9220 ohms


In [12]:
# ============================================================
# 10) Salvando uma versão processada dos dados
# ============================================================
# Aqui vamos adicionar a tensão prevista pelo modelo linear e
# salvar um novo arquivo CSV com essa informação.
# ============================================================

df["tensao_prevista_V"] = y_previsto

caminho_saida = "/content/drive/MyDrive/Colab Notebooks/dados_fisica_lei_ohm_interativo_processado.csv"
df.to_csv(caminho_saida, index=False)

print("Arquivo processado salvo em:")
print(caminho_saida)


Arquivo processado salvo em:
/content/drive/MyDrive/Colab Notebooks/dados_fisica_lei_ohm_interativo_processado.csv


## Sugestões para discutir em sala

Você pode explorar perguntas como:

- Por que os pontos não ficam todos exatamente sobre a reta?
- O que representa fisicamente o coeficiente angular?
- O que aconteceria se o componente não fosse ôhmico?
- Como identificar experimentalmente um resistor que não obedece à Lei de Ohm?
- O valor do intercepto deveria ser exatamente zero? Por quê?

Essas perguntas ajudam a conectar a análise computacional com a interpretação física.


## Exercício sugerido para os alunos

Peça aos alunos para:

1. abrir o arquivo;
2. calcular a resistência média;
3. comparar a resistência média com a resistência obtida pelo ajuste linear;
4. alterar o gráfico interativo para mostrar outra variável;
5. salvar um novo arquivo com uma coluna extra criada por eles.

Assim, além da física, eles praticam análise de dados em Python.
